# Explore SQL Window Functions

Window functions perform a calculation **across a set of rows** that are related to the current row, without collapsing those rows the way `GROUP BY` does.

```sql
function_name (expression) OVER (
    [PARTITION BY col1, col2, ...]
    [ORDER BY col3 [ASC|DESC], ...]
    [frame_clause]
)
```

## Core building blocks

| Clause | Purpose |
|--------|---------|
| `OVER ()` | The window — required |
| `PARTITION BY` | Split rows into independent groups (like GROUP BY, but rows stay visible) |
| `ORDER BY` | Define sequence inside each partition (needed for ranking, running totals, LAG/LEAD) |
| Frame (`ROWS` / `RANGE`) | Control exactly which rows around the current row are included |

## Categories you will practice

1. **Ranking** – `ROW_NUMBER`, `RANK`, `DENSE_RANK`
2. **Aggregate windows** – `SUM`, `AVG`, `COUNT`, `MIN`, `MAX` over a window
3. **Offset** – `LAG`, `LEAD`
4. **Value** – `FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`
5. **Frames** – running totals, moving averages
6. **Practical patterns** – top-N per group, gaps, comparisons to previous row

## Execution order reminder

```
FROM → WHERE → GROUP BY → HAVING → WINDOW → SELECT → ORDER BY → LIMIT
```

Window functions run after `HAVING` and before the final `SELECT` list is returned.


## Exercise 1 – Global row number

**Question:** Assign a sequential number to every invoice ordered by `InvoiceId`.

### Instructions
- Select `InvoiceId`, `CustomerId`, `Total`
- Add `ROW_NUMBER() OVER (ORDER BY InvoiceId) AS RowNum`
- Return the first 10 rows


**Hints**
```sql
SELECT InvoiceId, CustomerId, Total,
       ROW_NUMBER() OVER (ORDER BY InvoiceId) AS RowNum
FROM invoices
ORDER BY InvoiceId
LIMIT 10;
```
`OVER (ORDER BY …)` without `PARTITION BY` treats the whole result as one partition.


## Exercise 2 – PARTITION BY + ROW_NUMBER

**Question:** Number the invoices **within each billing country**, ordered by total amount descending.

### Instructions
- Select `BillingCountry`, `InvoiceId`, `Total`
- Add `ROW_NUMBER() OVER (PARTITION BY BillingCountry ORDER BY Total DESC) AS rn`
- Order the output by country and rn
- Limit to 15 rows so you can see several countries


**Hints**
`PARTITION BY` restarts the numbering for each country.  
This is the foundation of “top-N per group” queries.


## Exercise 3 – RANK vs DENSE_RANK vs ROW_NUMBER

**Question:** Compare the three ranking functions on invoice totals for the USA.

### Instructions
Filter to `BillingCountry = 'USA'`, then compute:

- `ROW_NUMBER() OVER (ORDER BY Total DESC)`
- `RANK() OVER (ORDER BY Total DESC)`
- `DENSE_RANK() OVER (ORDER BY Total DESC)`

Observe how ties are handled differently.


**Hints – difference summary**

| Function | Tie behavior | Next value after a tie |
|----------|--------------|------------------------|
| `ROW_NUMBER` | Always unique 1,2,3… | Continues sequentially |
| `RANK` | Same rank for ties | Skips numbers |
| `DENSE_RANK` | Same rank for ties | Does **not** skip |


## Exercise 4 – Running total

**Question:** Compute a running total of invoice amounts ordered by `InvoiceId`.

### Instructions
- Select `InvoiceId`, `Total`
- Add `SUM(Total) OVER (ORDER BY InvoiceId) AS RunningTotal`
- Show the first 12 rows


**Hints**
When you use an aggregate with `ORDER BY` inside `OVER`, SQLite defaults to a frame of  
`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — i.e. a classic running total.


## Exercise 5 – Running total per partition

**Question:** For each billing country, compute a running total of invoice amounts ordered by invoice date.

### Instructions
- Partition by `BillingCountry`
- Order by `InvoiceDate`, `InvoiceId` (to make the order deterministic)
- Show country, date, total, and the running total
- Limit to a couple of countries for readability (e.g. filter `BillingCountry IN ('Canada','France')`)


**Hints**
```sql
SUM(Total) OVER (
    PARTITION BY BillingCountry
    ORDER BY InvoiceDate, InvoiceId
) AS CountryRunningTotal
```


## Exercise 6 – LAG and LEAD

**Question:** For each invoice (ordered by InvoiceId), show the previous and next invoice totals.

### Instructions
- Select `InvoiceId`, `Total`
- `LAG(Total) OVER (ORDER BY InvoiceId) AS PrevTotal`
- `LEAD(Total) OVER (ORDER BY InvoiceId) AS NextTotal`
- Also compute the difference from the previous total
- Show first 10 rows


**Hints**
- `LAG(col)` = value from the previous row
- `LEAD(col)` = value from the next row
- Both return `NULL` when there is no previous/next row
- Optional: `LAG(Total, 1, 0)` supplies a default instead of NULL


## Exercise 7 – FIRST_VALUE / LAST_VALUE

**Question:** For every invoice in the USA, show:
- the highest total in the USA (`FIRST_VALUE` after ordering by Total DESC)
- how far the current total is from that maximum

### Instructions
Filter to USA, order by Total descending, and use `FIRST_VALUE(Total) OVER (…)`.


**Hints**
```sql
FIRST_VALUE(Total) OVER (
    PARTITION BY BillingCountry
    ORDER BY Total DESC
) AS MaxTotalInCountry
```
`LAST_VALUE` often needs an explicit frame (`ROWS BETWEEN … AND UNBOUNDED FOLLOWING`) to behave as people expect.


## Exercise 8 – Moving average (frame clause)

**Question:** Compute a 3-invoice moving average of totals ordered by InvoiceId.

### Instructions
Use an explicit frame:

```sql
AVG(Total) OVER (
    ORDER BY InvoiceId
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
) AS MovingAvg3
```

Show InvoiceId, Total, and the moving average for the first 15 rows.


**Hints – common frames**

| Frame | Meaning |
|-------|---------|
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | Running total / avg from start |
| `ROWS BETWEEN N PRECEDING AND CURRENT ROW` | Last N+1 rows |
| `ROWS BETWEEN N PRECEDING AND N FOLLOWING` | Sliding window centered on current row |
| `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING` | From current row to the end |


## Exercise 9 – Top-N per group (practical pattern)

**Question:** Return the **2 most expensive invoices** for each billing country.

### Instructions
1. CTE with `ROW_NUMBER() OVER (PARTITION BY BillingCountry ORDER BY Total DESC)`
2. Outer query keeps `rn <= 2`
3. Order by country and rn


**Hints**
This is the standard, portable “top-N per group” pattern you already saw in the LATERAL and HAVING notebooks. Window functions make it clean and efficient.


## Exercise 10 – Challenge: Year-over-year style comparison

**Question:** For each customer, order their invoices by date and show:
- the current invoice total
- the previous invoice total (`LAG`)
- the difference (current − previous)
- a running total of that customer’s spend

### Instructions
Partition all window functions by `CustomerId` and order by `InvoiceDate`, `InvoiceId`.  
Filter to a few customers (e.g. CustomerId ≤ 5) so the result is readable.


**Hints**
You can put multiple window functions in the same SELECT, each with its own `OVER` clause (or reuse a named window if you prefer).

```sql
LAG(Total) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate, InvoiceId)
SUM(Total) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate, InvoiceId)
```
